# Computer Vision Benchmark — Arduino UNO Q (4GB RAM)

Notebook chấm điểm model CV ứng viên cho **Edge Agentic Assistant**, chia
làm 4 phần độc lập theo đúng các tác vụ đã chốt:

1. **Image classification** — MobileNetV3 / EfficientNet-Lite trên subset ImageNet-1k
2. **Object detection** — YOLOv8n / YOLO11n / NanoDet trên COCO128 (mAP)
3. **OCR** — PaddleOCR-lite / Tesseract / TrOCR-small trên text in tổng hợp (CER)
4. **Image captioning / VQA** — GIT-base / SmolVLM / Florence-2 trên COCO Captions (BLEU)

**Insight quan trọng cho UNO Q (4GB, chia sẻ cho cả SLM+STT+TTS+CV):**
chạy 4 model chuyên biệt riêng cho 4 tác vụ trên gần như chắc chắn KHÔNG
vừa ngân sách RAM cùng lúc với SLM+STT+TTS. **Florence-2-base (0.23B)**
là model vision-language multi-task làm được cả detection + OCR +
captioning trong MỘT model — phần 4 sẽ đo trực tiếp để so trade-off
"1 model đa năng" so với "4 model chuyên biệt".

## 0. Chuẩn bị môi trường

In [1]:
%pip uninstall torch -y -q
%pip install torch --index-url https://download.pytorch.org/whl/cu124 -q
%pip install -q "transformers>=4.44.0" "datasets>=2.20.0" accelerate timm ultralytics \
    pillow psutil pandas matplotlib sacrebleu jiwer \
    sentencepiece protobuf pytesseract rapidocr-onnxruntime

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.29.1 requires torch>=2.14.0, but you have torch 2.6.0+cu124 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch, platform
print("Đang chạy trên:", platform.system(), platform.machine())
print("CUDA available:", torch.cuda.is_available(), "| torch:", torch.__version__)
if not torch.cuda.is_available():
    print("⚠️  Không thấy GPU — các model phần 4 (captioning/VLM) sẽ chậm trên CPU.")
else:
    print("GPU:", torch.cuda.get_device_name(0))

Đang chạy trên: Windows AMD64
CUDA available: False | torch: 2.14.1+cpu
⚠️  Không thấy GPU — các model phần 4 (captioning/VLM) sẽ chậm trên CPU.


In [3]:
import threading, time, gc, json, os
from pathlib import Path
import psutil

RESULTS_DIR = Path("./results_cv")
RESULTS_DIR.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class RamSampler:
    """Giống hệt bản dùng trong stt_benchmark.ipynb — xem ghi chú ở đó."""

    def __init__(self, interval: float = 0.3):
        self.interval = interval
        self._samples = []
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._proc = psutil.Process(os.getpid())

    def _run(self):
        while not self._stop.is_set():
            self._samples.append(self._proc.memory_info().rss / (1024 * 1024))
            time.sleep(self.interval)

    def start(self):
        self._thread.start()

    def stop(self):
        self._stop.set()
        self._thread.join(timeout=2)

    @property
    def peak_mb(self):
        return max(self._samples) if self._samples else 0.0

    @property
    def avg_mb(self):
        return sum(self._samples) / len(self._samples) if self._samples else 0.0


def save_row(subdir: str, name: str, row: dict):
    d = RESULTS_DIR / subdir
    d.mkdir(exist_ok=True)
    with open(d / f"{name}.json", "w") as f:
        json.dump(row, f, indent=2)


def load_row(subdir: str, name: str):
    p = RESULTS_DIR / subdir / f"{name}.json"
    return json.load(open(p)) if p.exists() else None

In [4]:
from huggingface_hub import login
login()


---
## 1. Image Classification

ImageNet-1k validation **yêu cầu đăng nhập HF** (gated dataset, giống
Gemma ở bản SLM) — chấp nhận license trên trang HF rồi `huggingface-cli
login` trước khi chạy. Dùng subset (`EVAL_SAMPLES`) để chạy nhanh khi
thử nghiệm; tăng lên full 50,000 ảnh khi chốt số liệu báo cáo.

In [5]:
CLS_MODELS = [
    {"name": "MobileNetV3-Small", "timm_name": "mobilenetv3_small_100.lamb_in1k", "params_b": 0.0025},
    {"name": "MobileNetV3-Large", "timm_name": "mobilenetv3_large_100.ra_in1k", "params_b": 0.0054},
    {"name": "EfficientNet-Lite0", "timm_name": "efficientnet_lite0.ra_in1k", "params_b": 0.0047},
]

CLS_EVAL_SAMPLES = 1000  # tăng lên 50000 (full val) khi chốt số liệu báo cáo


def load_imagenet_subset(n: int):
    """Dùng repo chính chủ "ILSVRC/imagenet-1k" (không phải alias "imagenet-1k"
    cũ) — alias cũ dùng loading script kiểu cũ, không tương thích với
    datasets>=4.0 (lỗi HfUriError, cùng gốc với lỗi gặp ở stt_benchmark.ipynb
    khi dùng "librispeech_asr" thay vì "openslr/librispeech_asr"). Repo
    ILSVRC đã ở dạng Parquet nên không cần trust_remote_code hay chỉ định
    revision đặc biệt.
    """
    from datasets import load_dataset
    ds = load_dataset("ILSVRC/imagenet-1k", split="validation", streaming=True)
    return list(ds.take(n))


def measure_classifier_one(entry: dict, samples: list) -> dict:
    import timm
    from PIL import Image

    gc.collect()
    sampler = RamSampler()
    sampler.start()
    t0 = time.time()

    model = timm.create_model(entry["timm_name"], pretrained=True).to(DEVICE).eval()
    cfg = timm.data.resolve_data_config({}, model=model)
    transform = timm.data.create_transform(**cfg)

    correct, t_infer0 = 0, time.time()
    with torch.no_grad():
        for s in samples:
            img = s["image"].convert("RGB")
            x = transform(img).unsqueeze(0).to(DEVICE)
            pred = model(x).argmax(dim=1).item()
            if pred == s["label"]:
                correct += 1
    infer_s = time.time() - t_infer0
    load_s = t_infer0 - t0
    sampler.stop()

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return {
        "model": entry["name"], "params_b": entry["params_b"],
        "load_s": round(load_s, 2), "ram_avg_mb": round(sampler.avg_mb, 1),
        "ram_peak_mb": round(sampler.peak_mb, 1),
        "top1_acc": round(correct / len(samples), 4),
        "ms_per_image": round(infer_s / len(samples) * 1000, 1),
        "n_samples": len(samples),
    }


print("Tải subset ImageNet-1k validation (cần HF_TOKEN đã login + chấp nhận license)...")
cls_samples = load_imagenet_subset(CLS_EVAL_SAMPLES)
print(f"Đã có {len(cls_samples)} ảnh.")

cls_rows = []
for entry in CLS_MODELS:
    cached = load_row("classification", entry["name"])
    if cached:
        cls_rows.append(cached)
        continue
    try:
        row = measure_classifier_one(entry, cls_samples)
        save_row("classification", entry["name"], row)
        print(f"[done] {row}")
        cls_rows.append(row)
    except Exception as e:
        print(f"[ERROR] {entry['name']} thất bại: {e}")
        cls_rows.append({"model": entry["name"], "error": str(e)})

import pandas as pd
cls_df = pd.DataFrame(cls_rows)
cls_df

Tải subset ImageNet-1k validation (cần HF_TOKEN đã login + chấp nhận license)...


Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/294 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Đã có 1000 ảnh.


,model,params_b,load_s,ram_avg_mb,ram_peak_mb,top1_acc,ms_per_image,n_samples
0,MobileNetV3-Small,0.0025,3.10,1949.4,2345.2,0.685,8.1,1000
1,MobileNetV3-Large,0.0054,3.09,1871.6,1884.3,0.764,11.7,1000
2,EfficientNet-Lite0,0.0047,3.20,1880.2,1908.3,0.757,12.3,1000


---
## 2. Object Detection

Dùng **COCO128** (128 ảnh, bộ dataset nhỏ chính thức của Ultralytics
dùng để kiểm tra nhanh) thay vì COCO val2017 đầy đủ (~5000 ảnh, ~1GB) —
mAP trên COCO128 chỉ mang tính **so sánh tương đối giữa các model trong
cùng notebook này**, không phải con số mAP chính thức để so với leaderboard
bên ngoài. Ghi rõ điều này nếu đưa vào báo cáo.

In [6]:
DET_MODELS = [
    {"name": "YOLOv8n", "weights": "yolov8n.pt", "params_b": 0.0032},
    {"name": "YOLO11n", "weights": "yolo11n.pt", "params_b": 0.0026},
    {"name": "YOLOv8s", "weights": "yolov8s.pt", "params_b": 0.0113},
]


def measure_detector_one(entry: dict) -> dict:
    from ultralytics import YOLO

    gc.collect()
    sampler = RamSampler()
    sampler.start()
    t0 = time.time()

    model = YOLO(entry["weights"])
    load_s = time.time() - t0

    t_val0 = time.time()
    metrics = model.val(data="coco128.yaml", device=0 if DEVICE == "cuda" else "cpu", verbose=False)
    val_s = time.time() - t_val0
    sampler.stop()

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return {
        "model": entry["name"], "params_b": entry["params_b"],
        "load_s": round(load_s, 2), "ram_avg_mb": round(sampler.avg_mb, 1),
        "ram_peak_mb": round(sampler.peak_mb, 1),
        "map50_95": round(float(metrics.box.map), 4),
        "map50": round(float(metrics.box.map50), 4),
        "val_s": round(val_s, 1),
        "note": "mAP trên COCO128 — tương đối trong notebook này, không phải số COCO val2017 chính thức",
    }


det_rows = []
for entry in DET_MODELS:
    cached = load_row("detection", entry["name"])
    if cached:
        det_rows.append(cached)
        continue
    try:
        row = measure_detector_one(entry)
        save_row("detection", entry["name"], row)
        print(f"[done] {row}")
        det_rows.append(row)
    except Exception as e:
        print(f"[ERROR] {entry['name']} thất bại: {e}")
        det_rows.append({"model": entry["name"], "error": str(e)})

det_df = pd.DataFrame(det_rows)
det_df

,model,params_b,load_s,ram_avg_mb,ram_peak_mb,map50_95,map50,val_s,note
0,YOLOv8n,0.0032,1.74,2107.3,2543.8,0.4454,0.6054,9.5,mAP trên COCO128 — tương đối trong notebook nà...
1,YOLO11n,0.0026,1.38,2399.7,2670.2,0.5024,0.6696,4.7,mAP trên COCO128 — tương đối trong notebook nà...
2,YOLOv8s,0.0113,2.19,2538.9,2899.3,0.5855,0.7585,9.4,mAP trên COCO128 — tương đối trong notebook nà...


---
## 3. OCR

Dùng **text in tổng hợp** (render chuỗi biết trước bằng PIL thành ảnh,
font/nền/nhiễu đa dạng) thay vì dataset scene-text ngoài đời — nhanh,
không cần tải dataset, đo đúng khả năng engine đọc chữ IN RÕ (nhãn/đồng
hồ đo trong phòng lab thường là chữ in, không phải chữ viết tay/ảnh chụp
góc nghiêng). **Giới hạn:** không phản ánh được điều kiện thực tế khó
hơn (ánh sáng, góc chụp, chữ mờ) — nên bổ sung vài ảnh chụp thật nếu có
thiết bị để kiểm tra chéo trước khi chốt model.

In [7]:
OCR_MODELS = [
    {"name": "Tesseract", "kind": "tesseract"},
    {"name": "PaddleOCR-mobile-en", "kind": "paddleocr"},
    {"name": "TrOCR-small-printed", "kind": "trocr", "hf_repo": "microsoft/trocr-small-printed"},
]

OCR_TEST_STRINGS = [
    "TEMP: 23.5 C", "VOLTAGE 12.0V", "PRESSURE 1.2 BAR", "SN-884213-A",
    "BATCH 2026-10-03", "RPM 3450", "CH1 ON  CH2 OFF", "pH 7.02",
    "FLOW RATE 1.5 L/MIN", "ERROR CODE E04",
]


def render_text_image(text: str):
    from PIL import Image, ImageDraw, ImageFont
    img = Image.new("RGB", (400, 100), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("DejaVuSansMono.ttf", 28)
    except OSError:
        font = ImageFont.load_default()
    draw.text((10, 30), text, fill="black", font=font)
    return img


def _ocr_tesseract(img):
    import pytesseract
    pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
    return pytesseract.image_to_string(img, config="--psm 7").strip()


def _ocr_paddleocr(img, engine_holder: dict):
    # Giữ tên hàm/kind "paddleocr" để không phải sửa phần còn lại; engine thật là RapidOCR.
    import numpy as np
    if "engine" not in engine_holder:
        from rapidocr_onnxruntime import RapidOCR
        engine_holder["engine"] = RapidOCR()
    result, _ = engine_holder["engine"](np.array(img))
    if not result:
        return ""
    return " ".join(r[1] for r in result)


def _ocr_trocr(img, model, processor):
    pixel_values = processor(images=img, return_tensors="pt").pixel_values.to(DEVICE)
    ids = model.generate(pixel_values, max_new_tokens=32)
    return processor.batch_decode(ids, skip_special_tokens=True)[0]


def char_error_rate(ref: str, hyp: str) -> float:
    import jiwer
    return jiwer.cer(ref, hyp)


def measure_ocr_one(entry: dict, images_and_text: list) -> dict:
    gc.collect()
    sampler = RamSampler()
    sampler.start()
    t0 = time.time()

    engine_holder = {}
    trocr_model = trocr_proc = None
    if entry["kind"] == "trocr":
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel
        trocr_proc = TrOCRProcessor.from_pretrained(entry["hf_repo"])
        trocr_model = VisionEncoderDecoderModel.from_pretrained(entry["hf_repo"]).to(DEVICE)
    load_s = time.time() - t0

    cers, t_infer0 = [], time.time()
    for img, text in images_and_text:
        if entry["kind"] == "tesseract":
            hyp = _ocr_tesseract(img)
        elif entry["kind"] == "paddleocr":
            hyp = _ocr_paddleocr(img, engine_holder)
        else:
            hyp = _ocr_trocr(img, trocr_model, trocr_proc)
        cers.append(char_error_rate(text, hyp))
    infer_s = time.time() - t_infer0
    sampler.stop()

    if trocr_model is not None:
        del trocr_model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return {
        "model": entry["name"], "load_s": round(load_s, 2),
        "ram_avg_mb": round(sampler.avg_mb, 1), "ram_peak_mb": round(sampler.peak_mb, 1),
        "cer": round(sum(cers) / len(cers), 4),
        "ms_per_image": round(infer_s / len(images_and_text) * 1000, 1),
        "n_samples": len(images_and_text),
    }


ocr_samples = [(render_text_image(t), t) for t in OCR_TEST_STRINGS]

ocr_rows = []
for entry in OCR_MODELS:
    cached = load_row("ocr", entry["name"])
    if cached:
        ocr_rows.append(cached)
        continue
    try:
        row = measure_ocr_one(entry, ocr_samples)
        save_row("ocr", entry["name"], row)
        print(f"[done] {row}")
        ocr_rows.append(row)
    except Exception as e:
        print(f"[ERROR] {entry['name']} thất bại: {e}")
        ocr_rows.append({"model": entry["name"], "error": str(e)})

ocr_df = pd.DataFrame(ocr_rows)
ocr_df

[ERROR] TrOCR-small-printed thất bại: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.


,model,load_s,ram_avg_mb,ram_peak_mb,cer,ms_per_image,n_samples,error
0,Tesseract,0.0,2153.7,2160.7,0.1772,145.8,10.0,NaN
1,PaddleOCR-mobile-en,0.0,2323.3,2546.7,0.0432,387.3,10.0,NaN
2,TrOCR-small-printed,NaN,NaN,NaN,NaN,NaN,NaN,Couldn't instantiate the backend tokenizer fro...


---
## 4. Image Captioning / VQA — và so sánh với Florence-2 đa năng

Chấm BLEU-4 trên subset COCO Captions cho khả năng mô tả ảnh. **Florence-2**
được đo riêng ở cả 3 tác vụ (detection/OCR/captioning) bằng CÙNG MỘT model
đã nạp — đây là phép so sánh "1 model đa năng" so với tổng RAM của 3 model
chuyên biệt ở các phần 2-3 cộng lại.

In [ ]:
CAP_MODELS = [
    {"name": "GIT-base", "hf_repo": "microsoft/git-base-coco"},
    {"name": "SmolVLM-256M-Instruct", "hf_repo": "HuggingFaceTB/SmolVLM-256M-Instruct"},
    {"name": "Florence-2-base", "hf_repo": "florence-community/Florence-2-base"},
]

CAP_EVAL_SAMPLES = 100


def load_coco_caption_subset(n: int):
    """"HuggingFaceM4/COCO" dùng loading script kiểu cũ (cùng vấn đề đã gặp
    với librispeech_asr/imagenet-1k) — chỉ định revision="refs/convert/parquet"
    để dùng nhánh Parquet tự động mà HF tạo cho mọi dataset, tồn tại độc
    lập với trạng thái loading script ở branch main. Ở nhánh này,
    "2014_captions" là data_dir, không phải config (name=) — bản Parquet
    chỉ còn 1 BuilderConfig "default" (giống lỗi "BuilderConfig 'clean'
    not found" đã gặp ở stt_benchmark.ipynb với librispeech_asr).
    """
    from datasets import load_dataset
    ds = load_dataset(
        "HuggingFaceM4/COCO", data_dir="2014_captions", split="validation",
        streaming=True, revision="refs/convert/parquet",
    )
    return list(ds.take(n))


def _load_vlm(repo: str):
    from transformers import AutoProcessor, AutoModelForCausalLM
    if "florence" in repo.lower():
        from transformers import Florence2ForConditionalGeneration
        proc = AutoProcessor.from_pretrained(repo)
        model = Florence2ForConditionalGeneration.from_pretrained(repo)
        return proc, model.to(DEVICE).eval()

    proc = AutoProcessor.from_pretrained(repo, trust_remote_code=True)
    try:
        from transformers import AutoModelForImageTextToText as AutoVLM
    except ImportError:
        from transformers import AutoModelForVision2Seq as AutoVLM
    try:
        model = AutoVLM.from_pretrained(repo, trust_remote_code=True)
    except Exception:  # GIT đăng ký qua CausalLM
        model = AutoModelForCausalLM.from_pretrained(repo, trust_remote_code=True)
    return proc, model.to(DEVICE).eval()    


def _caption(repo: str, proc, model, img) -> str:
    repo = repo.lower()
    with torch.no_grad():
        if "florence" in repo:
            inp = proc(text="<CAPTION>", images=img, return_tensors="pt").to(DEVICE)
            ids = model.generate(**inp, max_new_tokens=50, num_beams=1, do_sample=False)
            return proc.batch_decode(ids, skip_special_tokens=True)[0].strip()
        if "smolvlm" in repo:
            msgs = [{"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": "Describe this image in one short sentence."}]}]
            prompt = proc.apply_chat_template(msgs, add_generation_prompt=True)
            inp = proc(text=prompt, images=[img], return_tensors="pt").to(DEVICE)
            ids = model.generate(**inp, max_new_tokens=50)
            return proc.batch_decode(ids[:, inp["input_ids"].shape[1]:],
                                     skip_special_tokens=True)[0].strip()
        inp = proc(images=img, return_tensors="pt").to(DEVICE)  # GIT
        ids = model.generate(pixel_values=inp["pixel_values"], max_new_tokens=50)
        return proc.batch_decode(ids, skip_special_tokens=True)[0].strip()


def _get_refs(s: dict, k: int = 5) -> list:
    sents = s.get("sentences")
    if isinstance(sents, dict):
        raw = list(sents["raw"])
    elif isinstance(sents, list):
        raw = [x["raw"] if isinstance(x, dict) else x for x in sents]
    else:
        raw = [s.get("caption", "")]
    raw = raw[:k]
    while len(raw) < k:  # sacrebleu cần số reference bằng nhau cho mọi ảnh
        raw.append(raw[-1])
    return raw


def measure_captioner_one(entry: dict, samples: list) -> dict:
    import sacrebleu

    gc.collect()
    sampler = RamSampler()
    sampler.start()
    t0 = time.time()
    proc, model = _load_vlm(entry["hf_repo"])
    load_s = time.time() - t0

    hyps, refs, t_infer0 = [], [], time.time()
    for s in samples:
        img = s["image"].convert("RGB")
        hyps.append(_caption(entry["hf_repo"], proc, model, img))
        refs.append(_get_refs(s))
    infer_s = time.time() - t_infer0
    sampler.stop()

    bleu = sacrebleu.corpus_bleu(hyps, [list(r) for r in zip(*refs)]).score

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return {
        "model": entry["name"], "load_s": round(load_s, 2),
        "ram_avg_mb": round(sampler.avg_mb, 1), "ram_peak_mb": round(sampler.peak_mb, 1),
        "bleu4": round(bleu, 2),
        "ms_per_image": round(infer_s / len(samples) * 1000, 1),
        "n_samples": len(samples),
    }


print("Tải subset COCO Captions...")
cap_samples = load_coco_caption_subset(CAP_EVAL_SAMPLES)

cap_rows = []
for entry in CAP_MODELS:
    cached = load_row("captioning", entry["name"])
    if cached:
        cap_rows.append(cached)
        continue
    try:
        row = measure_captioner_one(entry, cap_samples)
        save_row("captioning", entry["name"], row)
        print(f"[done] {row}")
        cap_rows.append(row)
    except Exception as e:
        print(f"[ERROR] {entry['name']} thất bại: {e}")
        cap_rows.append({"model": entry["name"], "error": str(e)})

cap_df = pd.DataFrame(cap_rows)
cap_df

Tải subset COCO Captions...


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/2.26k [00:00<?, ?B/s]

c:\Users\Minor\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Minor\.cache\huggingface\hub\models--florence-community--Florence-2-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


preprocessor_config.json:   0%|          | 0.00/603 [00:00<?, ?B/s]

...

c:\Users\Minor\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Minor\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  463MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/665 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/292 [00:00<?, ?B/s]

## 5. Tổng hợp: 4 model chuyên biệt vs 1 Florence-2 đa năng

In [ ]:
# So tổng RAM của "đội hình chuyên biệt" tốt nhất (1 model/tác vụ, không
# tính classification vì Florence-2 không làm classification) với RAM của
# riêng Florence-2-base (dùng chung cho cả 3 tác vụ còn lại).
best_det = det_df.loc[det_df["map50_95"].idxmax()] if "map50_95" in det_df.columns and not det_df.empty else None
best_ocr = ocr_df.loc[ocr_df["cer"].idxmin()] if "cer" in ocr_df.columns and not ocr_df.empty else None
florence_row = cap_df[cap_df["model"] == "Florence-2-base"]

print("=== Đội hình chuyên biệt (detection + OCR), RAM cộng dồn ===")
if best_det is not None and best_ocr is not None:
    specialized_ram = best_det["ram_peak_mb"] + best_ocr["ram_peak_mb"]
    print(f"  {best_det['model']}: {best_det['ram_peak_mb']} MB (mAP50-95={best_det['map50_95']})")
    print(f"  {best_ocr['model']}: {best_ocr['ram_peak_mb']} MB (CER={best_ocr['cer']})")
    print(f"  TỔNG: {specialized_ram:.0f} MB cho 2 model riêng (chưa tính captioning)")

print("\n=== Florence-2-base, 1 model làm cả 3 ===")
if not florence_row.empty:
    print(florence_row.to_string(index=False))
    print("  → so trực tiếp với TỔNG ở trên để quyết định kiến trúc.")

=== Đội hình chuyên biệt (detection + OCR), RAM cộng dồn ===
  YOLOv8s: 2899.3 MB (mAP50-95=0.5855)
  PaddleOCR-mobile-en: 2546.7 MB (CER=0.0432)
  TỔNG: 5446 MB cho 2 model riêng (chưa tính captioning)

=== Florence-2-base, 1 model làm cả 3 ===
          model  load_s  ram_avg_mb  ram_peak_mb  bleu4  ms_per_image  n_samples                                                                   error
Florence-2-base     NaN         NaN          NaN    NaN           NaN        NaN 'Florence2LanguageConfig' object has no attribute 'forced_bos_token_id'
  → so trực tiếp với TỔNG ở trên để quyết định kiến trúc.


## Lưu ý khi viết vào báo cáo

- **mAP trên COCO128** chỉ dùng để so sánh tương đối các model detection
  TRONG notebook này — không trích dẫn như số mAP COCO val2017 chính thức
  khi so với paper/leaderboard khác.
- **CER trên text in tổng hợp** không phản ánh điều kiện chụp thực tế
  (góc nghiêng, ánh sáng yếu, chữ mờ) — chỉ là sàn năng lực lý tưởng.
  Nếu có UNO Q + camera thật, nên chụp thêm ảnh nhãn/đồng hồ đo thực tế
  để kiểm tra chéo trước khi chốt model OCR.
- **BLEU-4** đo độ khớp n-gram với caption tham chiếu — không đo được
  caption "đúng nhưng diễn đạt khác" (BLEU sẽ phạt oan). Cân nhắc đọc
  qua một số caption mẫu bằng mắt thường trước khi kết luận chỉ dựa vào
  điểm số.
- Việc dùng Florence-2 thay 3 model chuyên biệt **đánh đổi**: thường kém
  chính xác hơn một chút ở từng tác vụ riêng lẻ, đổi lấy RAM thấp hơn
  nhiều khi cần chạy đồng thời với SLM+STT+TTS trên cùng 4GB. Ghi rõ
  đây là quyết định kiến trúc có chủ đích, không phải model yếu hơn do
  nhầm lẫn khi chọn.